# Train models from models/

Run from the repository root or `train` folder. Choose `MODEL_NAME` to train one model, or `None` for all three sequentially. Settings come from `train/cfg/v3.yaml`; checkpoints and metrics are saved per run.

On RunPod, use CUDA-enabled PyTorch and upload the dataset images and labels. See [README.md](README.md) for setup and [SOURCES.md](custom_modules/SOURCES.md) for the experimental blocks and aliases. Knowledge distillation is not included.


## Install the custom environment

Locate the repository and keep Ultralytics settings inside the project.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()

if ROOT.name == "train":
    ROOT = ROOT.parent

SETUP_SCRIPT = ROOT / "train" / "setup_ultralytics.py"

if not SETUP_SCRIPT.is_file():
    raise FileNotFoundError(
        f"Setup script not found: {SETUP_SCRIPT}. "
        "Open the notebook from your repository root or train folder."
    )

os.environ.setdefault(
    "YOLO_CONFIG_DIR",
    str(ROOT / ".cache/ultralytics")
)

print("Repository:", ROOT)
print("Python:", sys.executable)

Repository: /root/lightweight-cnn
Python: /usr/bin/python3.11


Install the pinned Ultralytics source and dependencies. After the first installation, restart the kernel before Setup.


In [2]:
subprocess.run(
    [sys.executable, str(SETUP_SCRIPT)],
    check=True,
    cwd=str(ROOT)
)

ULTRALYTICS_SOURCE = ROOT / "vendor" / "ultralytics"

if not (
    ULTRALYTICS_SOURCE / "ultralytics" / "__init__.py"
).is_file():
    raise FileNotFoundError(
        f"Custom Ultralytics source not found: {ULTRALYTICS_SOURCE}"
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-e",
        str(ULTRALYTICS_SOURCE),
        "timm==1.0.31",
        "wandb",
        "nbformat",
    ],
    check=True,
    cwd=str(ROOT)
)

print("Installation completed successfully!")
print("Restart the kernel before continuing to Cell 3.")

Prepared Ultralytics v8.3.0 with custom layers in /root/lightweight-cnn/vendor/ultralytics
Obtaining file:///root/lightweight-cnn/vendor/ultralytics
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for ultralytics (pyproject.toml): started
  Building editable for ultralytics (pyproject.toml): finished with status 'done'
  Created wheel for ultralytics: filename=ultralytics-8.3.0-0.editable-py3-none-any.whl size=22771 sha256=9a197d16ca8d914ba65a0c2788f0925391809717f7d1f0c5930145cea527eb8a
  Stored in directory: /tmp/pip-ephem-wh


[notice] A new release of pip is available: 26.2 -> 26.3
[notice] To update, run: python3.11 -m pip install --upgrade pip


## Setup

Run these cells in order. To train another model, change `MODEL_NAME` and rerun the remaining Setup cells, architecture check, and training. Each setup preserves earlier runs in separate directories.


Locate the repository after a kernel restart.


In [4]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()

if ROOT.name == "train":
    ROOT = ROOT.parent

os.environ.setdefault(
    "YOLO_CONFIG_DIR",
    str(ROOT / ".cache/ultralytics")
)

CUSTOM_ULTRALYTICS = (
    ROOT / "vendor" / "ultralytics"
).resolve()

PACKAGE_INIT = (
    CUSTOM_ULTRALYTICS / "ultralytics" / "__init__.py"
)

if not PACKAGE_INIT.is_file():
    raise FileNotFoundError(
        f"Custom Ultralytics package missing: {PACKAGE_INIT}. "
        "Run the installation cells first."
    )

# Add the actual checkout directory to Python's import path.
source_path = str(CUSTOM_ULTRALYTICS)

if source_path in sys.path:
    sys.path.remove(source_path)

sys.path.insert(0, source_path)

# Prevent an already imported, incorrect package from being reused.
previous = sys.modules.get("ultralytics")

if previous is not None:
    previous_file = getattr(previous, "__file__", None)

    if (
        previous_file is None
        or not Path(previous_file).resolve().is_relative_to(
            CUSTOM_ULTRALYTICS
        )
    ):
        raise RuntimeError(
            "A different Ultralytics version is already loaded. "
            "Restart the kernel and run Cell 3 again."
        )

print("Python:", sys.executable)
print("Custom Ultralytics:", CUSTOM_ULTRALYTICS)

Python: /usr/bin/python3.11
Custom Ultralytics: /root/lightweight-cnn/vendor/ultralytics


Import the training and reporting libraries.


In [5]:
import csv
import gc

from datetime import datetime
from time import perf_counter

try:
    import ultralytics

except ModuleNotFoundError as exc:
    if exc.name != "ultralytics":
        raise

    raise ModuleNotFoundError(
        f"Ultralytics could not be imported using {sys.executable}. "
        "Check the custom installation and restart the kernel."
    ) from exc

import torch
import wandb
import yaml

from IPython.display import display
from ultralytics import YOLO, settings

print("All training libraries imported successfully!")

All training libraries imported successfully!


/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Verify the custom installation. CUDA should report `True` on a GPU RunPod.


In [6]:
actual_source = Path(ultralytics.__file__).resolve()

if not actual_source.is_relative_to(CUSTOM_ULTRALYTICS):
    raise RuntimeError(
        f"Wrong Ultralytics installation loaded: {actual_source}. "
        f"Expected source under: {CUSTOM_ULTRALYTICS}"
    )

print("Ultralytics:", ultralytics.__version__)
print("Ultralytics source:", actual_source)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Warning: CUDA GPU is not available.")

Ultralytics: 8.3.0
Ultralytics source: /root/lightweight-cnn/vendor/ultralytics/ultralytics/__init__.py
PyTorch: 2.15.0+cu132
CUDA available: True
GPU: NVIDIA GeForce RTX 4090


Choose one model, or set `MODEL_NAME = None` for all three. Keep the space in the ADown filename.


In [ ]:
# MODEL_NAME = "yolov8_baseline.yaml"
# MODEL_NAME = "ADown _C2fFaster.yaml"
MODEL_NAME = "GSConv_VoV-GSCP1.yaml"
# MODEL_NAME = None

WANDB_PROJECT = "cone-detection"
CONFIG = ROOT / "train/cfg/v3.yaml"

Check that the selected architecture file exists.


In [34]:
MODEL_PATHS = sorted(
    path for path in (ROOT / "models").iterdir()
    if path.is_file() and path.suffix.lower() in {".yaml", ".yml"}
)
if not MODEL_PATHS:
    raise FileNotFoundError(f"No model architecture YAMLs in {ROOT / 'models'}")
if MODEL_NAME is not None:
    available_names = [path.name for path in MODEL_PATHS]
    if MODEL_NAME not in available_names:
        raise ValueError(f"Unknown MODEL_NAME {MODEL_NAME!r}. Choose one of {available_names}, or None for all.")
    MODEL_PATHS = [ROOT / "models" / MODEL_NAME]

Load training and export settings from `v3.yaml`. Keep settings consistent across comparisons; use a fixed batch size if needed. See [README.md](README.md) for worker settings.


In [35]:
train_args = yaml.safe_load(CONFIG.read_text(encoding="utf-8"))
export_args = {key: train_args.pop(key) for key in ("format", "half", "int8", "nms")}
IMGSZ = train_args["imgsz"]

Create a timestamped output directory for this run.


In [36]:
experiment_name = train_args.pop("name", None) or "comparison"
RUN_PROJECT = ROOT / train_args["project"] / f"{experiment_name}_{datetime.now():%Y%m%d_%H%M%S_%f}"
RUN_PROJECT.mkdir(parents=True, exist_ok=False)
train_args["project"] = str(RUN_PROJECT)

Read the dataset YAML and resolve its local root.


In [38]:
SOURCE_DATA = ROOT / train_args["data"]
if not SOURCE_DATA.is_file():
    runpod_data = Path("/workspace/roboflow-test-letterboxed-2880x2880/data.yaml")
    if not runpod_data.is_file():
        raise FileNotFoundError(
            f"Dataset YAML missing at {SOURCE_DATA} and {runpod_data}. "
            "Set data in train/cfg/v3.yaml to your dataset YAML."
        )
    SOURCE_DATA = runpod_data
print(f"Dataset YAML: {SOURCE_DATA}")
dataset_config = yaml.safe_load(SOURCE_DATA.read_text(encoding="utf-8"))
DATASET_ROOT = SOURCE_DATA.parent.resolve()
dataset_config["path"] = str(DATASET_ROOT)

Dataset YAML: /workspace/roboflow-test-letterboxed-2880x2880/data.yaml


Verify the training and validation paths. Upload missing images and labels before continuing.


In [39]:
for split in ("train", "val"):
    paths = dataset_config[split]
    for split_path in paths if isinstance(paths, list) else [paths]:
        if not (DATASET_ROOT / split_path).exists():
            raise FileNotFoundError(f"Missing {split} data: {DATASET_ROOT / split_path}. Upload the dataset first.")

Save a portable dataset YAML for this run, preserving the original.


In [40]:
DATA = RUN_PROJECT / "data.yaml"
DATA.write_text(yaml.safe_dump(dataset_config, sort_keys=False), encoding="utf-8")
train_args["data"] = str(DATA)


Define a helper to save `comparison.csv` after each model.


In [41]:
SUMMARY_PATH = RUN_PROJECT / "comparison.csv"
SUMMARY_FIELDS = ["model", "best_weights", "training_seconds", "validation_seconds", "mAP50", "mAP50-95"]


def save_comparison(records):
    with SUMMARY_PATH.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=SUMMARY_FIELDS)
        writer.writeheader()
        writer.writerows(records)

Review the selected models and output directory.


In [42]:
print("Models:", *(path.name for path in MODEL_PATHS), sep="\n  ")
print("Outputs:", RUN_PROJECT)

Models:
  ADown _C2fFaster.yaml
Outputs: /root/lightweight-cnn/runs/comparison_20261122_105807_748869


## Check selected architectures

Build each selected model before training. Resolve any reported errors first. For a short training and checkpoint test, see [README.md](README.md).


In [43]:
# Clear approval on every attempt, including attempts that fail partway through.
CHECKED_MODEL_PATHS = None
load_errors = []
for model_path in MODEL_PATHS:
    try:
        candidate = YOLO(str(model_path), task="detect")
        print(f"OK: {model_path.name}")
        del candidate
    except Exception as error:
        load_errors.append(f"{model_path.name}: {type(error).__name__}: {error}")
    finally:
        gc.collect()

if load_errors:
    raise RuntimeError(
        "Training has not started. These architectures could not load:\n  "
        + "\n  ".join(load_errors)
        + "\nRerun the installation cell, restart the kernel, and rerun Setup and this check."
    )
CHECKED_MODEL_PATHS = tuple((str(path), path.read_bytes()) for path in MODEL_PATHS)
print("All architectures loaded successfully. Ready to train.")


OK: ADown _C2fFaster.yaml
All architectures loaded successfully. Ready to train.


## Train selected models

Train from scratch with validation enabled. Run these cells in order; the training loop starts the run.


Sign in to W&B for a separate logging run per model.


In [44]:
settings.update({"wandb": True})
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.forge.coreweave.com. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://forge.coreweave.com/wandb/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: No netrc file found, creating one.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /root/.netrc
wandb: Currently logged in as: yuma-iwamoto1 (yuma-iwamoto1-university-of-toronto) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


True

Define the training function. It collects metrics and releases model memory; training has not started yet.


In [45]:
def train_one_model(model_path):
    run_name = model_path.stem.replace(" ", "_")
    start = perf_counter()
    print(f"Training {model_path.name}")
    with wandb.init(project=WANDB_PROJECT, name=run_name, dir=str(RUN_PROJECT)):
        model = YOLO(str(model_path), task="detect")
        results = model.train(**{**train_args, "name": run_name})
        record = {
            "model": model_path.stem,
            "best_weights": str(model.trainer.best),
            "training_seconds": round(perf_counter() - start, 1),
        }
        if results is not None:
            record.update({"mAP50": float(results.box.map50), "mAP50-95": float(results.box.map)})
    del model, results
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return record

Start training and save metrics after each model. This starts fresh training; rerun Setup first for a new output directory.


In [49]:
train_args.pop("cutmix", None)

print("Training arguments updated.")
print("cutmix present:", "cutmix" in train_args)

Training arguments updated.
cutmix present: False


In [50]:
if globals().get("CHECKED_MODEL_PATHS") != tuple((str(path), path.read_bytes()) for path in MODEL_PATHS):
    raise RuntimeError("Run the architecture check successfully after selecting or editing models.")

run_records = []
best_weights = {}
suite_start = perf_counter()
for model_path in MODEL_PATHS:
    record = train_one_model(model_path)
    best_weights[model_path.stem] = Path(record["best_weights"])
    run_records.append(record)
    save_comparison(run_records)
    print(record)

Training ADown _C2fFaster.yaml


New https://pypi.org/project/ultralytics/8.4.216 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.0 🚀 Python-3.11.18 torch-2.15.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4090, 24083MiB)
engine/trainer: task=detect, mode=train, model=/root/lightweight-cnn/models/ADown _C2fFaster.yaml, data=/root/lightweight-cnn/runs/comparison_20261122_105807_748869/data.yaml, epochs=100, time=None, patience=100, batch=-1, imgsz=960, save=True, save_period=-1, cache=True, device=None, workers=0, project=/root/lightweight-cnn/runs/comparison_20261122_105807_748869, name=ADown__C2fFaster, exist_ok=False, pretrained=True, optimizer=SGD, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, sour

100%|██████████| 755k/755k [00:00<00:00, 1.48MB/s]


                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      1344  ultralytics.nn.modules.block.ADown           [16, 32]                      
  2                  -1  1      3920  ultralytics.nn.modules.custom_blocks.C2f_Faster[32, 32, 1, True]             
  3                  -1  1      5248  ultralytics.nn.modules.block.ADown           [32, 64]                      
  4                  -1  2     22144  ultralytics.nn.modules.custom_blocks.C2f_Faster[64, 64, 2, True]             
  5                  -1  1     20736  ultralytics.nn.modules.block.ADown           [64, 128]                     
  6                  -1  2     87552  ultralytics.nn.modules.custom_blocks.C2f_Faster[128, 128, 2, True]           
  7                  -1  1     82432  ultralytics.nn.modules.block.ADown         

ADown _C2fFaster summary: 254 layers, 1,576,248 parameters, 1,576,232 gradients, 3.0 GFLOPs

Freezing layer 'model.22.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks with YOLOv8n...


100%|██████████| 6.25M/6.25M [00:00<00:00, 184MB/s]


... (207 outputs left out)

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 37/37 [00:28<00:00,  1.28it/s]


                   all       2898      53002      0.766      0.576      0.634       0.41
             blue_cone       2393      21326      0.863      0.675      0.761      0.483
     large_orange_cone        785       2050      0.793       0.75       0.79      0.576
           orange_cone        951       5275      0.803      0.645      0.705      0.462
          unknown_cone        191       1720      0.527      0.144      0.167     0.0645
           yellow_cone       2433      22631      0.845      0.667      0.745      0.462
Speed: 0.2ms preprocess, 1.0ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /root/lightweight-cnn/runs/comparison_20261122_105807_748869/ADown__C2fFaster


lr/pg0,█▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
lr/pg1,▃████▇▇▇▇▆▆▆▆▆▆▆▆▅▅▅▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁
lr/pg2,▃███▇▇▇▆▆▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁
metrics/mAP50(B),▁▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇████████████████
metrics/mAP50-95(B),▁▄▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇██████████████
metrics/precision(B),▁▃▆▆▇▇▇▇█████████▅▆▆▆▅▆▆▆▆▅▅▅▅▅▅▅▅▅▅▅▅▅▆
metrics/recall(B),▁▃▄▅▅▆▆▆▆▆▆▆▆▆▆▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇████████
model/GFLOPs,▁
model/parameters,▁
model/speed_PyTorch(ms),▁
+6,...


{'model': 'ADown _C2fFaster', 'best_weights': '/root/lightweight-cnn/runs/comparison_20261122_105807_748869/ADown__C2fFaster/weights/best.pt', 'training_seconds': 25308.6, 'mAP50': 0.6337824048940678, 'mAP50-95': 0.4095917333754955}


In [51]:
from pathlib import Path
from ultralytics import YOLO

run = Path("/root/lightweight-cnn/runs/comparison_20261122_093707_304354")
model = YOLO(str(run / "yolov8_baseline/weights/last.pt"))

results = model.train(
    data=str(run / "data.yaml"),
    epochs=50,
    imgsz=960,
    batch=48,
    device=0,
    optimizer="SGD",
    lr0=0.001,  # Example starting learning rate for fine-tuning
    project=str(run),
    name="yolov8_baseline_finetune_20",
)

FileNotFoundError: [Errno 2] No such file or directory: '/root/lightweight-cnn/runs/comparison_20261122_093707_304354/yolov8_baseline/weights/last.pt'

Show training time and comparison records.


In [52]:
print(f"Training total: {(perf_counter() - suite_start) / 3600:.2f} hours")
print("Comparison:", SUMMARY_PATH)
display(run_records)

Training total: 7.48 hours
Comparison: /root/lightweight-cnn/runs/comparison_20261122_105807_748869/comparison.csv


[{'model': 'ADown _C2fFaster',
  'best_weights': '/root/lightweight-cnn/runs/comparison_20261122_105807_748869/ADown__C2fFaster/weights/best.pt',
  'training_seconds': 25308.6,
  'mAP50': 0.6337824048940678,
  'mAP50-95': 0.4095917333754955}]

## Select a checkpoint

Default to the first model trained in this run. Change `SELECTED_MODEL` for another model, or set `BEST_WEIGHTS` to an earlier checkpoint after Setup. Custom checkpoints require the same layer definitions.


In [53]:
SELECTED_MODEL = next(iter(best_weights))
BEST_WEIGHTS = best_weights[SELECTED_MODEL]
best_model = YOLO(str(BEST_WEIGHTS), task="detect")
print(BEST_WEIGHTS)


/root/lightweight-cnn/runs/comparison_20261122_105807_748869/ADown__C2fFaster/weights/best.pt


## Separate validation (optional)

Reevaluate each best checkpoint and update `comparison.csv`. Use `split="test"` for final evaluation and save those results separately.


In [54]:
for record in run_records:
    start = perf_counter()
    validation_model = YOLO(record["best_weights"], task="detect")
    metrics = validation_model.val(
        data=str(DATA), split="val", imgsz=IMGSZ,
        workers=train_args["workers"],
        project=str(Path(record["best_weights"]).parent.parent), name="validation",
    )
    record.update({
        "mAP50": float(metrics.box.map50),
        "mAP50-95": float(metrics.box.map),
        "validation_seconds": round(perf_counter() - start, 1),
    })
    save_comparison(run_records)
    print(record)
    del validation_model, metrics
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Ultralytics 8.3.0 🚀 Python-3.11.18 torch-2.15.0+cu132 CUDA:0 (NVIDIA GeForce RTX 4090, 24083MiB)
ADown _C2fFaster summary (fused): 215 layers, 1,571,816 parameters, 0 gradients, 2.9 GFLOPs


val: Scanning /workspace/roboflow-test-letterboxed-2880x2880/valid/labels.cache... 2898 images, 13 backgrounds, 0 corrupt: 100%|██████████| 2898/2898 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 182/182 [01:23<00:00,  2.17it/s]


                   all       2898      53002      0.767      0.578      0.637      0.417
             blue_cone       2393      21326      0.868      0.678      0.766      0.496
     large_orange_cone        785       2050      0.792      0.749       0.79      0.579
           orange_cone        951       5275      0.805      0.648      0.706      0.471
          unknown_cone        191       1720      0.522      0.144      0.171     0.0666
           yellow_cone       2433      22631       0.85       0.67       0.75      0.474
Speed: 0.2ms preprocess, 1.8ms inference, 0.0ms loss, 0.5ms postprocess per image
Results saved to /root/lightweight-cnn/runs/comparison_20261122_105807_748869/ADown__C2fFaster/validation
{'model': 'ADown _C2fFaster', 'best_weights': '/root/lightweight-cnn/runs/comparison_20261122_105807_748869/ADown__C2fFaster/weights/best.pt', 'training_seconds': 25308.6, 'mAP50': 0.6366672443154427, 'mAP50-95': 0.4172910124792654, 'validation_seconds': 86.0}


Review the validation metrics saved to the comparison CSV.


In [55]:
display(run_records)

[{'model': 'ADown _C2fFaster',
  'best_weights': '/root/lightweight-cnn/runs/comparison_20261122_105807_748869/ADown__C2fFaster/weights/best.pt',
  'training_seconds': 25308.6,
  'mAP50': 0.6366672443154427,
  'mAP50-95': 0.4172910124792654,
  'validation_seconds': 86.0}]

## TensorRT export

Export FP16 `best.engine` beside `best.pt` using `v3.yaml` and batch size 1. Requires an NVIDIA GPU and [compatible TensorRT](https://docs.ultralytics.com/integrations/tensorrt/); build on deployment-compatible hardware.

For calibrated INT8, set `half: False` and `int8: True`.


In [ ]:
start = perf_counter()
ENGINE_PATH = Path(best_model.export(
    **export_args,
    data=str(DATA),
    imgsz=IMGSZ,
    batch=1,
    device=0,
))
print(f"Engine: {ENGINE_PATH}")
print(f"Export: {perf_counter() - start:.1f} s")


## Image prediction

Choose `IMAGE_PATH`; the default uses a test image.


In [ ]:
IMAGE_PATH = next((DATASET_ROOT / "test/images").glob("*.jpg"))

Predict with the selected engine and save the annotated image.


In [ ]:
start = perf_counter()
engine = YOLO(str(ENGINE_PATH), task="detect")
predictions = engine.predict(
    source=str(IMAGE_PATH),
    imgsz=IMGSZ,
    conf=0.4,
    device=0,
    save=True,
    project=str(BEST_WEIGHTS.parent.parent),
    name="predictions",
)

Display the prediction and timing, including engine loading.


In [ ]:
print(f"Prediction (including engine loading): {perf_counter() - start:.1f} s")
for prediction in predictions:
    display(prediction.plot(pil=True))

## TensorRT benchmark

Evaluate the engine on validation images with batch size 1 and square inputs.


In [ ]:
start = perf_counter()
engine = YOLO(str(ENGINE_PATH), task="detect")
benchmark_metrics = engine.val(
    data=str(DATA),
    split="val",
    imgsz=IMGSZ,
    batch=1,
    conf=0.001,
    device=0,
    workers=train_args["workers"],
    rect=False,
    plots=False,
)

Show accuracy, timing, and inference FPS. FPS excludes preprocessing and postprocessing.


In [ ]:
print(f"mAP50-95: {benchmark_metrics.box.map:.4f}")
print("Milliseconds per image:", benchmark_metrics.speed)
print(f"Inference FPS: {1000 / benchmark_metrics.speed['inference']:.1f}")
print(f"Benchmark: {perf_counter() - start:.1f} s")